# Machina Interactive Tour

15-minute hands-on tour of Machina's capabilities. Build an agent, ask questions, trigger workflows, switch CMMS backends, and define custom workflows.

**Prerequisites:** `pip install machina-ai[litellm]` and Ollama running (or set `OPENAI_API_KEY`).

In [ ]:
import sys
from pathlib import Path

# Ensure machina is importable when running from the examples folder
repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd().parent.parent
sys.path.insert(0, str(repo_root / "src"))

sample_dir = Path.cwd() / "sample_data" if (Path.cwd() / "sample_data").exists() else Path.cwd().parent / "sample_data"

## 1. Build an Agent

13 lines. CMMS data + equipment manuals + LLM.

In [ ]:
from machina import Agent, Plant
from machina.connectors.cmms import GenericCmmsConnector
from machina.connectors.docs import DocumentStoreConnector

cmms = GenericCmmsConnector(data_dir=sample_dir / "cmms")
docs = DocumentStoreConnector(paths=[sample_dir / "manuals"])

agent = Agent(
    name="Tour Agent",
    plant=Plant(name="Demo Plant"),
    connectors=[cmms, docs],
    channels=[],  # no interactive channel in a notebook
    llm="ollama:llama3",  # or "openai:gpt-4o"
)

print(f"Agent: {agent.name} | LLM: {agent.llm} | Connectors: {len(agent.connectors)}")

In [ ]:
await agent.start()
print(f"Agent started. Assets loaded: {len(agent.plant.assets)}")

## 2. Ask Questions

The agent resolves entities, retrieves context from connectors, searches manuals via RAG, and answers with domain knowledge.

In [ ]:
# Equipment query -- agent resolves "P-201" to the actual asset
answer = await agent.handle_message(
    "What is the bearing replacement procedure for pump P-201?"
)
print(answer)

In [ ]:
# Diagnosis -- agent uses FailureAnalyzer + LLM reasoning
answer = await agent.handle_message(
    "Pump P-201 shows vibration of 7.8 mm/s (threshold 6.0) and "
    "bearing temperature of 64C (threshold 60C). "
    "What failure modes could explain this?"
)
print(answer)

In [ ]:
# Spare parts -- agent checks inventory via CMMS connector
answer = await agent.handle_message(
    "Which spare parts are compatible with pump P-201? Are they in stock?"
)
print(answer)

## 3. Trigger a Workflow

Register the built-in alarm-to-work-order workflow and trigger it in sandbox mode.

In [ ]:
from machina.workflows.builtins import alarm_to_workorder

# Register the workflow
agent.register_workflow(alarm_to_workorder)
agent.sandbox = True  # safe mode

print(f"Workflow: {alarm_to_workorder.name} ({len(alarm_to_workorder.steps)} steps)")
for i, step in enumerate(alarm_to_workorder.steps, 1):
    print(f"  {i}. {step.name} -> {step.action}")

In [ ]:
# Trigger with a simulated alarm
result = await agent.trigger_workflow(alarm_to_workorder.name, {
    "alarm_id": "ALM-001",
    "asset_id": "P-201",
    "parameter": "vibration_velocity_mm_s",
    "value": 7.8,
    "threshold": 6.0,
    "severity": "warning",
})

print(f"Result: {'SUCCESS' if result.success else 'FAILED'} ({result.duration_seconds:.2f}s)")
for sr in result.steps:
    icon = "+" if sr.success else "~" if sr.skipped else "x"
    print(f"  [{icon}] {sr.name}")

## 4. Switch CMMS Backend

Same agent logic, different CMMS. In production you'd swap the connector:

In [ ]:
# In production, swap one line:
#
#   from machina.connectors import SapPM
#   cmms = SapPM(url="https://sap.company.com/odata/v4", auth=...)
#
#   from machina.connectors import Maximo
#   cmms = Maximo(url="https://maximo.company.com/oslc", auth=...)
#
# The rest stays identical:
#   agent = Agent(connectors=[cmms, docs], llm="openai:gpt-4o")

# For this demo, we ask the same question -- works with any backend
answer = await agent.handle_message("List all critical assets with their equipment class.")
print(answer)

## 5. Define a Custom Workflow

Build your own workflow from scratch using the DSL.

In [ ]:
from machina.workflows import Workflow, Step, Trigger, TriggerType, ErrorPolicy

my_workflow = Workflow(
    name="Quick Inspection",
    description="Check an asset and recommend next steps.",
    trigger=Trigger(type=TriggerType.MANUAL),
    steps=[
        Step("get_asset",
             action="cmms.read_assets",
             inputs={"asset_id": "{trigger.asset_id}"},
             on_error=ErrorPolicy.STOP),
        Step("check_history",
             action="cmms.read_maintenance_history",
             inputs={"asset_id": "{trigger.asset_id}"},
             on_error=ErrorPolicy.SKIP),
        Step("recommend",
             action="agent.reason",
             prompt=(
                 "Asset: {get_asset}\n"
                 "History: {check_history}\n\n"
                 "Recommend: next maintenance action and priority."
             )),
    ],
)

agent.register_workflow(my_workflow)
print(f"Registered: {my_workflow.name} ({len(my_workflow.steps)} steps)")

In [ ]:
result = await agent.trigger_workflow("Quick Inspection", {"asset_id": "P-201"})

print(f"Result: {'SUCCESS' if result.success else 'FAILED'}")
for sr in result.steps:
    print(f"  {sr.name}: {str(sr.output)[:200] if sr.output else 'skipped'}")

In [ ]:
await agent.stop()
print("Tour complete!")

## What's Next?

| Example | What you'll learn |
|---------|-------------------|
| [quickstart/](quickstart/) | Interactive CLI agent |
| [01_alarm_response/](01_alarm_response/) | Built-in alarm workflow |
| [02_predictive_pipeline/](02_predictive_pipeline/) | Full 10-step autonomous pipeline |
| [03_cmms_portability/](03_cmms_portability/) | Same agent, different CMMS |
| [04_custom_workflows/](04_custom_workflows/) | Build your own workflows |